In [ ]:
# Configure every external directory here before executing the notebook.
from pathlib import Path
import os
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent
RAW_DIR = Path(os.environ.get('RIDESHARE_RAW_DIR', str(PROJECT_DIR / 'data/raw/fhvhv')))
PROCESSED_DIR = Path(os.environ.get('RIDESHARE_PROCESSED_DIR', str(PROJECT_DIR / 'data/processed')))
ZONE_DIR = Path(os.environ.get('RIDESHARE_ZONE_DIR', str(PROJECT_DIR / 'data/reference/taxi_zones')))
WEATHER_DIR = Path(os.environ.get('RIDESHARE_WEATHER_DIR', str(PROJECT_DIR / 'data/reference/weather_hourly')))
ANALYSIS_DIR = PROJECT_DIR / 'outputs/demand_analysis'
# Spark users: configure JAVA_HOME and, on Windows when required, HADOOP_HOME externally.

import os
import glob

# Use the Java/Hadoop installation configured in your environment.
# Use the Java/Hadoop installation configured in your environment.

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName('TLC_FHVHV_Processing') \
    .master('local[*]') \
    .config('spark.driver.memory', '8g') \
    .config('spark.sql.parquet.enableVectorizedReader', 'true') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')

print(f'Spark version: {spark.version}')
print('Environment ready!')

    # .config('spark.sql.shuffle.partitions', '8') \

# 02 — Data Processing & Aggregation: FHVHV Trip Records (2024–2025)

**Project:** NYC TLC Ridesharing Analysis Pipeline  
**Phase:** 1 — Data Foundation  
**Goal:** Clean raw FHVHV data in memory, engineer features, and aggregate to output files

**Local directories:** configure paths in the first code cell.
**Local directories:** configure paths in the first code cell.

---

### What this notebook does:
1. Set up PySpark environment
2. Load raw FHVHV parquet files (2024–2025)
3. Clean data (nulls, invalid values, outliers)
4. Engineer features (datetime, fare, wait time columns)
5. Aggregate to 5 output files:
   - `fhvhv_by_zone_month.parquet` — all metrics by zone × month
   - `fhvhv_by_hour.parquet` — all metrics by date × hour
   - `fhvhv_by_zone_hour.parquet` — all metrics by zone × date × hour
   - `fhvhv_by_15min.parquet` — wait time by date × 15min interval
   - `fhvhv_by_zone_15min.parquet` — wait time by zone × date × 15min interval

## 0. Environment Setup

## 1. Define Paths

In [ ]:
RAW_DATA_PATH = str(RAW_DIR)
OUTPUT_PATH = str(PROCESSED_DIR)

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Policy cutoff date
POLICY_DATE = '2025-01-05'

# Load 2024-2025 parquet files
TARGET_YEARS = ['2024', '2025']
all_files = glob.glob(os.path.join(RAW_DATA_PATH, 'fhvhv_*.parquet'))
target_files = sorted([
    f for f in all_files
    if any(year in os.path.basename(f) for year in TARGET_YEARS)
])

print(f'Files to load: {len(target_files)}')
print(f'Output path:   {OUTPUT_PATH}')

## 2. Load Raw Data

In [ ]:
print('Loading raw FHVHV parquet files...')
# df = spark.read.parquet(*target_files)


df = None
for file in target_files:
    temp_df = spark.read.parquet(file)

    if df is None:
        df = temp_df
    else:
        df = df.unionByName(temp_df,allowMissingColumns=True)
    print(f"loaded:{file} | Rows: {temp_df.count()}")

raw_count = df.count()
print(f'Raw rows loaded: {raw_count:,}')

## 3. Data Cleaning

Rules:
- Check the times for all records with null `cbd_congestion_fee` and fill null with 0 if they are during the pre-policy period (per-policy is not implemented then)
- Remove 16 duplicates
- Remove rows where `trip_miles <= 0` (invalid distance)
- Remove rows where `trip_time <= 0` (invalid duration)
- Remove rows where `base_passenger_fare <= 0` (invalid fare)
- Remove rows where `pickup_datetime < request_datetime` (impossible)
- Remove extreme outliers: `trip_miles` > 200, `trip_time` > 10800 (3 hours), `speed` > 120, or `speed` < 1

In [ ]:
df.printSchema()

In [ ]:
df = df\
    .withColumn('trip_time_cal',F.unix_timestamp('dropoff_datetime') - F.unix_timestamp('pickup_datetime')) \
    .withColumn('trip_speed',(F.col('trip_miles')/F.col('trip_time'))*3600)

In [ ]:
# # check in records with null cbd_congestion_price 

# print('Null values after policy:',df.filter((F.col('cbd_congestion_fee').isNull()) & (F.col('pickup_datetime') >= '2025-01-05')).count())

# print('None-null values before policy:', df.filter((F.col('cbd_congestion_fee').isNotNull()) & (F.col('pickup_datetime') < '2025-01-05')).count())

# # Check the actual earliest date
# df.filter(F.col('cbd_congestion_fee').isNotNull()) \
#   .agg(F.min('pickup_datetime').alias('first_congestion_charge')) \
#   .show(truncate=False)

In [ ]:
# # Verify the exact cutoff with a few days around Jan 1

# df.filter(F.col('cbd_congestion_fee').isNotNull()) \
#   .withColumn('date', F.to_date('pickup_datetime')) \
#   .groupBy('date') \
#   .count() \
#   .orderBy('date') \
#   .show(20)

In [ ]:
df_clean = df \
    .dropDuplicates() \
    .filter(F.col('trip_miles') > 0) \
    .filter(F.col('trip_time_cal') > 0) \
    .filter(F.col('base_passenger_fare') > 0) \
    .filter(F.col('trip_speed') > 1) \
    .filter(F.col('driver_pay') > 0) \
    .filter(F.col('pickup_datetime') >= F.col('request_datetime')) \
    .filter(F.col('trip_miles') <= 200) \
    .filter(F.col('trip_time_cal') <= 10800) \
    .filter(F.col('trip_speed') <= 120) \
    .withColumn('cbd_congestion_fee',
        F.when(F.col('cbd_congestion_fee').isNull(), 0)
         .otherwise(F.col('cbd_congestion_fee')))

clean_count = df_clean.count()
removed = raw_count - clean_count
print(f'Rows after cleaning: {clean_count:,}')
print(f'Rows removed:        {removed:,} ({removed/raw_count*100:.2f}%)')

## 4. Feature Engineering

In [ ]:
df_feat = df_clean \
    .withColumn('year',         F.year('pickup_datetime')) \
    .withColumn('month',        F.month('pickup_datetime')) \
    .withColumn('week',         F.weekofyear('pickup_datetime')) \
    .withColumn('date',         F.to_date('pickup_datetime')) \
    .withColumn('hour',         F.hour('pickup_datetime')) \
    .withColumn('day_of_week',  F.dayofweek('pickup_datetime')) \
    .withColumn('is_weekend',   F.when(F.dayofweek('pickup_datetime').isin([1, 7]), 1).otherwise(0)) \
    .withColumn('15min_interval', (F.minute('pickup_datetime') / 15).cast('integer') * 15) \
    .withColumn('policy_period',
        F.when(F.col('pickup_datetime') < F.lit(POLICY_DATE), 'pre_policy')
         .otherwise('post_policy')) \
    .withColumn('wait_time_sec',
        F.unix_timestamp('pickup_datetime') - F.unix_timestamp('request_datetime')) \
    .withColumn('trip_duration_min', F.col('trip_time_cal') / 60) \
    .withColumn('total_fare',
        F.col('base_passenger_fare') + F.col('tolls') + F.col('tips') +
        F.col('congestion_surcharge') + F.col('sales_tax') + F.col('bcf') +
        F.col('airport_fee') + F.col('cbd_congestion_fee')) \
    .withColumn('tnc_comp',
        F.col('base_passenger_fare') + F.col('tips') - F.col('driver_pay'))

# Cache for repeated aggregations
df_feat.cache()
print('Feature engineering complete!')
new_cols = ['year', 'month', 'week', 'date', 'hour', 'day_of_week', 'is_weekend',
            '15min_interval', 'policy_period', 'wait_time_sec',
            'trip_duration_min', 'total_fare', 'tnc_comp']
print('New columns added:', new_cols)
# for c in new_cols:
#     print(f'  + {c}')

## 5. Define Aggregation Helper

Using aggregate-first approach:
- Sum numerators and denominators separately
- Then divide to get rates

In [ ]:
def compute_metrics(df, group_cols):
    """
    Aggregate all metrics using aggregate-first approach.
    Group by group_cols and compute all metrics.
    """
    return df.groupBy(group_cols).agg(
        # Demand
        F.count('*').alias('trip_count'),

        # Intermediate sums for rate calculations
        F.sum('wait_time_sec').alias('_sum_wait_time'),
        F.sum('base_passenger_fare').alias('_sum_base_fare'),
        F.sum('total_fare').alias('_sum_total_fare'),
        F.sum('driver_pay').alias('_sum_driver_pay'),
        F.sum('tnc_comp').alias('_sum_tnc_comp'),
        F.sum('trip_miles').alias('_sum_trip_miles'),
        F.sum('trip_duration_min').alias('_sum_trip_duration_min')
    ).withColumn(
        'avg_wait_time_sec',     F.round(F.col('_sum_wait_time') / F.col('trip_count'), 2)
    ).withColumn(
        'base_fare_per_mile',    F.round(F.col('_sum_base_fare') / F.col('_sum_trip_miles'), 4)
    ).withColumn(
        'base_fare_per_min',     F.round(F.col('_sum_base_fare') / F.col('_sum_trip_duration_min'), 4)
    ).withColumn(
        'base_fare_per_trip',    F.round(F.col('_sum_base_fare') / F.col('trip_count'), 4)
    ).withColumn(
        'total_fare_per_trip',   F.round(F.col('_sum_total_fare') / F.col('trip_count'), 4)
    ).withColumn(
        'driver_pay_per_mile',   F.round(F.col('_sum_driver_pay') / F.col('_sum_trip_miles'), 4)
    ).withColumn(
        'driver_pay_per_min',    F.round(F.col('_sum_driver_pay') / F.col('_sum_trip_duration_min'), 4)
    ).withColumn(
        'driver_pay_per_trip',   F.round(F.col('_sum_driver_pay') / F.col('trip_count'), 4)
    ).withColumn(
        'tnc_commission_rate',   F.round(F.col('_sum_tnc_comp') / F.col('_sum_base_fare'), 4)
    ).drop(
        '_sum_wait_time', '_sum_base_fare', '_sum_total_fare',
        '_sum_driver_pay', '_sum_tnc_comp', '_sum_trip_miles', '_sum_trip_duration_min'
    )


def compute_waittime_only(df, group_cols):
    """
    Aggregate wait time only — used for 15min granularity files.
    """
    return df.groupBy(group_cols).agg(
        F.count('*').alias('trip_count'),
        F.sum('wait_time_sec').alias('_sum_wait_time')
    ).withColumn(
        'avg_wait_time_sec', F.round(F.col('_sum_wait_time') / F.col('trip_count'), 2)
    ).drop('_sum_wait_time')


print('Aggregation helpers defined!')

## 6. Aggregation & Save Output Files

### 6.1 fhvhv_by_zone_month.parquet
All metrics aggregated by `PULocationID × year × month`

In [ ]:
# print('Aggregating by zone × month...')

# for year in [2024, 2025]:
#     print(f'Processing year {year}...')
#     df_year = df_feat.filter(F.col('year') == year)
#     group_cols = ['PULocationID', 'year', 'month', 'policy_period']
#     df_zone_month = compute_metrics(df_year, group_cols) \
#         .orderBy('PULocationID', 'year', 'month')
    
#     output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_zone_month_{year}.parquet')
#     df_zone_month.write.mode('overwrite').parquet(output_file)
#     print(f'Saved: {output_file}')
#     print(f'Rows:  {df_zone_month.count():,}')

### 6.2 fhvhv_by_hour.parquet
All metrics aggregated by `date × hour`

In [ ]:
print('Aggregating by date × hour...')

for year in [2024, 2025]:
    print(f"Processing year {year}...")
    df_year = df_feat.filter(F.col('year') == year)
    
    group_cols = ['date', 'year', 'month', 'week', 'day_of_week', 'is_weekend', 'hour', 'policy_period']
    df_hour = compute_metrics(df_year, group_cols).orderBy('date', 'hour')

    output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_hour_{year}.parquet')
    df_hour.write.mode('overwrite').parquet(output_file)
    print(f'Saved: {output_file}')

    # Read back to verify instead of counting before saving
    df_check = spark.read.parquet(output_file)
    print(f'Rows: {df_check.count():,}')
    df_check.show(5)
    
    # Free memory before next year
    df_year.unpersist()
    df_hour.unpersist()

### 6.3 fhvhv_by_zone_hour.parquet
All metrics aggregated by `PULocationID × date × hour`

In [ ]:
print('Aggregating by zone × date × hour...')

for year in [2024,2025]:
    for month in range(1, 13):
        print(f'Processing {year}-{month}')
        df_month = df_feat.filter(
            (F.col('year') == year) & (F.col('month') == month)
        )
    
        group_cols = ['PULocationID', 'date', 'year', 'month', 'week', 'day_of_week', 'is_weekend', 'hour', 'policy_period']
        df_zone_hour = compute_metrics(df_month, group_cols).orderBy('PULocationID', 'date', 'hour')
        
        output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_zone_hour_{year}_{month:02d}.parquet')
        df_zone_hour.write.mode('overwrite').parquet(output_file)
        
        print(f'Saved: {output_file}')

        # Read back to verify instead of counting before saving
        df_check = spark.read.parquet(output_file)
        print(f'Rows: {df_check.count():,}')
        df_check.show(5)
        
        # Free memory before next year
        df_month.unpersist()
        df_zone_hour.unpersist()

### 6.4 fhvhv_by_15min.parquet
Wait time aggregated by `date × 15min interval`

In [ ]:
print('Aggregating wait time by date × 15min interval...')

for year in [2024,2025]:
    for month in range(1, 13):
        print(f'Processing {year}-{month}')
        df_month = df_feat.filter(
            (F.col('year') == year) & (F.col('month') == month)
        )
        group_cols = ['date', 'year', 'month', 'week', 'day_of_week', 'is_weekend', 'hour', '15min_interval', 'policy_period']
        df_15min = compute_waittime_only(df_month, group_cols).orderBy('date', 'hour', '15min_interval')
        
        output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_15min_{year}_{month:02d}.parquet')
        df_15min.write.mode('overwrite').parquet(output_file)
        
        print(f'Saved: {output_file}')

        # Read back to verify instead of counting before saving
        df_check = spark.read.parquet(output_file)
        print(f'Rows: {df_check.count():,}')
        df_check.show(5)
        
        # Free memory before next year
        df_month.unpersist()
        df_15min.unpersist()

### 6.5 fhvhv_by_zone_15min.parquet
Wait time aggregated by `PULocationID × date × 15min interval`

In [ ]:
print('Aggregating wait time by zone × date × 15min interval...')

for year in [2024,2025]:
    for month in range(1, 13):
        print(f'Processing {year}-{month}')
        df_month = df_feat.filter(
            (F.col('year') == year) & (F.col('month') == month)
        )

        group_cols = ['PULocationID', 'date', 'year', 'month', 'week', 'day_of_week', 'is_weekend', 'hour', '15min_interval', 'policy_period']
        df_zone_15min = compute_waittime_only(df_month, group_cols).orderBy('PULocationID', 'date', 'hour', '15min_interval')
        
        output_file = os.path.join(OUTPUT_PATH, f'fhvhv_by_zone_15min_{year}_{month:02d}.parquet')
        df_zone_15min.write.mode('overwrite').parquet(output_file)
        
        print(f'Saved: {output_file}')

        # Read back to verify instead of counting before saving
        df_check = spark.read.parquet(output_file)
        print(f'Rows: {df_check.count():,}')
        df_check.show(5)
        
        # Free memory before next year
        df_month.unpersist()
        df_zone_15min.unpersist()

## 7. Summary

In [ ]:
print('=' * 60)
print('PROCESSING COMPLETE — OUTPUT SUMMARY')
print('=' * 60)

output_files = [
    'fhvhv_by_zone_month.parquet',
    'fhvhv_by_hour.parquet',
    'fhvhv_by_zone_hour.parquet',
    'fhvhv_by_15min.parquet',
    'fhvhv_by_zone_15min.parquet',
    'fhvhv_by_OD_month.parquet'
]

for f in output_files:
    path = os.path.join(OUTPUT_PATH, f)
    if os.path.exists(path):
        size_mb = sum(
            os.path.getsize(os.path.join(path, ff))
            for ff in os.listdir(path)
            if ff.endswith('.parquet')
        ) / (1024 * 1024)
        print(f'  {f:45s} {size_mb:>8.1f} MB')

print()
print(f'Raw rows:      {raw_count:,}')
print(f'Cleaned rows:  {clean_count:,}')
print(f'Rows removed:  {removed:,} ({removed/raw_count*100:.2f}%)')
print()
print('Next step: 03a_eda_demand.ipynb')

---
## Notes

**Cleaning rules applied:**
- Removed null pickup/dropoff/request datetimes
- Removed trips with `trip_miles <= 0` or `trip_time <= 0`
- Removed trips with `base_passenger_fare <= 0`
- Removed trips where `pickup_datetime < request_datetime`
- Removed extreme outliers: `trip_miles > 200`, `trip_time > 10800s`
- Filled null `cbd_congestion_fee` with 0 (pre-policy trips)

**Aggregation method:** Aggregate-first (sum numerators and denominators separately before dividing)

**Output files:**
| File | Granularity | Metrics |
|---|---|---|
| `fhvhv_by_zone_month.parquet` | Zone × Month | All metrics |
| `fhvhv_by_hour.parquet` | Date × Hour | All metrics |
| `fhvhv_by_zone_hour.parquet` | Zone × Date × Hour | All metrics |
| `fhvhv_by_15min.parquet` | Date × 15min | Wait time only |
| `fhvhv_by_zone_15min.parquet` | Zone × Date × 15min | Wait time only |